# Lab 08-02: un RAG minimo

Lab: `corso/laboratori/lab-08-02-rag-minimo`

Esegui le celle in ordine (Runtime > Esegui tutto). Scarica fino a 1 GB e richiede alcuni minuti.


## 1. Preparazione

Scarica i binari ufficiali di llama.cpp (build `b11046`, quella del corso) e il materiale del lab.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # thread = core disponibili (oltre i core fisici si rallenta)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

## 2. Un server solo per due modelli

Il file `modelli.ini` descrive il modello di embedding (embeddinggemma, circa 330 MB) e il modello di chat; il server in *router mode* li carica a richiesta.

In [ ]:
%%bash
cat $LABS/lab-08-02-rag-minimo/modelli.ini

In [ ]:
!llama download -hf ggml-org/embeddinggemma-300M-GGUF:Q8_0 < /dev/null 2>&1 | tail -1
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF:Q8_0 < /dev/null 2>&1 | tail -1

In [ ]:
import subprocess, time, urllib.request
log = open("/tmp/llama-router.log", "w")
router = subprocess.Popen(["llama", "serve", "--host", "127.0.0.1", "--port", "18105", "--models-preset",
                           f"{LABS}/lab-08-02-rag-minimo/modelli.ini", "--models-max", "1", "-t", str(T)],
                          stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:18105/health", timeout=2); break
    except Exception:
        time.sleep(1)
print("router pronto")

In [ ]:
!pip -q install numpy requests

## 3. Indicizzare, cercare, rispondere

In [ ]:
%%bash
cd $LABS/lab-08-02-rag-minimo
export RAG_URL=http://127.0.0.1:18105
python3 rag_minimo.py indicizza
python3 rag_minimo.py cerca "Quanti bit per peso usa Q4_K_M?"
python3 rag_minimo.py rispondi "Quanti bit per peso usa Q4_K_M?"

In [ ]:
router.terminate(); router.wait()